
# Lab 5: Normalization, One‑Hot Encoding & Cyclic Encoding
**Student Name:** Shaheer Khan  
**Registration No:** 22JZELE0457  

This notebook prepares the feature‑engineered dataset for machine learning models by:
1. **Splitting** data into training (70%), validation (20%), and test (10%) sets.
2. **Normalizing** the target variable (`aep`) using Min‑Max scaling.
3. **One‑hot encoding** binary features (`holiday`, `weekend`).
4. **Cyclic encoding** periodic features (`hour`, `month`, `day_of_week`, `year_day`, and season indicators).
5. **Combining** all transformed features into final arrays and saving them as CSV files.

All paths follow the original notebook structure.



## 1. Imports and Data Loading
We load the feature‑extracted dataset from `5_features_extracted.csv` and inspect its structure.


In [1]:

import numpy as np
import pandas as pd
from pandas import read_csv
import pickle
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder
from timeseires.utils import t_v_t_split as sp

# Load the feature dataset
df = pd.read_csv(r'C:\Users\PMYLS\Documents\MachineLearningLaB\5_features_extracted.csv', 
                 index_col=['Datetime'], parse_dates=['Datetime'])
print("Data shape:", df.shape)
df.head()


Data shape: (121296, 11)


,aep,year_day,holiday,weekend,winter,spring,summer,fall,hour,month,day_of_week
Datetime,,,,,,,,,,,
2004-10-01 01:00:00,12379.0,275,0,0,0,0,0,1,1,10,4
2004-10-01 02:00:00,11935.0,275,0,0,0,0,0,1,2,10,4
2004-10-01 03:00:00,11692.0,275,0,0,0,0,0,1,3,10,4
2004-10-01 04:00:00,11597.0,275,0,0,0,0,0,1,4,10,4
2004-10-01 05:00:00,11681.0,275,0,0,0,0,0,1,5,10,4


In [2]:

# Check data types and missing values
df.info()
print("\nMissing values:\n", df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 121296 entries, 2004-10-01 01:00:00 to 2018-08-03 00:00:00
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   aep          121296 non-null  float64
 1   year_day     121296 non-null  int64  
 2   holiday      121296 non-null  int64  
 3   weekend      121296 non-null  int64  
 4   winter       121296 non-null  int64  
 5   spring       121296 non-null  int64  
 6   summer       121296 non-null  int64  
 7   fall         121296 non-null  int64  
 8   hour         121296 non-null  int64  
 9   month        121296 non-null  int64  
 10  day_of_week  121296 non-null  int64  
dtypes: float64(1), int64(10)
memory usage: 11.1 MB

Missing values:
 aep            0
year_day       0
holiday        0
weekend        0
winter         0
spring         0
summer         0
fall           0
hour           0
month          0
day_of_week    0
dtype: int64



## 2. Train / Validation / Test Split
We use the custom function `t_v_t_split` from the `timeseires.utils` module to split the data chronologically (70% train, 20% validation, 10% test). The split respects temporal order.


In [3]:

train_set, validation_set, test_set = sp.t_v_t(df, 70, 20, 10)

print("Train shape:", train_set.shape)
print("Validation shape:", validation_set.shape)
print("Test shape:", test_set.shape)


Train shape: (84907, 11)
Validation shape: (24259, 11)
Test shape: (12130, 11)


In [4]:

# Verify that the total rows match
total = train_set.shape[0] + validation_set.shape[0] + test_set.shape[0]
print("Total rows:", total)
print("Original rows:", df.shape[0])
assert total == df.shape[0], "Split totals do not match original data!"


Total rows: 121296
Original rows: 121296



## 3. Normalization of Target Variable
We scale the target `aep` using `MinMaxScaler` to the range [0, 1]. This helps models with gradient‑based learning. The scaler is fitted on the training set and then used to transform validation and test sets.


In [5]:

# Extract target values and reshape for scaler
train_set_load = train_set['aep'].values.reshape(-1, 1)
validation_set_load = validation_set['aep'].values.reshape(-1, 1)
test_set_load = test_set['aep'].values.reshape(-1, 1)

# Initialize and fit scaler on training data
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_set_load)

# Transform all sets
scaled_train_set_load = scaler.transform(train_set_load)
scaled_validation_set_load = scaler.transform(validation_set_load)
scaled_test_set_load = scaler.transform(test_set_load)

# Save scaler for later inverse transformation
pickle.dump(scaler, open("AEPscaler.pkl", 'wb'))

print("Scaled train shape:", scaled_train_set_load.shape)
print("Scaled validation shape:", scaled_validation_set_load.shape)
print("Scaled test shape:", scaled_test_set_load.shape)


Scaled train shape: (84907, 1)
Scaled validation shape: (24259, 1)
Scaled test shape: (12130, 1)



## 4. One‑hot Encoding of Binary Features
We apply one‑hot encoding to the binary features `holiday` (0/1) and `weekend` (0/1). Each becomes a two‑column one‑hot vector. The encoder is fitted on the training set and then used to transform validation and test sets.


In [6]:

# Training set
train_setO = train_set.values
holiday_train = train_setO[:, 2:3]      # column index of 'holiday'
weekend_train = train_setO[:, 3:4]      # column index of 'weekend'

# Initialize encoders
en_holiday = OneHotEncoder(handle_unknown='ignore')
en_weekend = OneHotEncoder(handle_unknown='ignore')

# Fit and transform training
holiday_train_encoded = en_holiday.fit_transform(holiday_train).toarray()
weekend_train_encoded = en_weekend.fit_transform(weekend_train).toarray()

# Combine
train_categorical = np.concatenate((holiday_train_encoded, weekend_train_encoded), axis=1)
print("Train categorical shape:", train_categorical.shape)

# Validation set
validation_setO = validation_set.values
holiday_val = validation_setO[:, 2:3]
weekend_val = validation_setO[:, 3:4]

holiday_val_encoded = en_holiday.transform(holiday_val).toarray()
weekend_val_encoded = en_weekend.transform(weekend_val).toarray()
validation_categorical = np.concatenate((holiday_val_encoded, weekend_val_encoded), axis=1)

# Test set
test_setO = test_set.values
holiday_test = test_setO[:, 2:3]
weekend_test = test_setO[:, 3:4]

holiday_test_encoded = en_holiday.transform(holiday_test).toarray()
weekend_test_encoded = en_weekend.transform(weekend_test).toarray()
test_categorical = np.concatenate((holiday_test_encoded, weekend_test_encoded), axis=1)

print("Validation categorical shape:", validation_categorical.shape)
print("Test categorical shape:", test_categorical.shape)


Train categorical shape: (84907, 4)
Validation categorical shape: (24259, 4)
Test categorical shape: (12130, 4)



## 5. Cyclic Encoding of Periodic Features
Periodic features (hour, day_of_week, month, year_day, seasons) are encoded as sine and cosine components to preserve their circular nature. This avoids the arbitrary discontinuity that raw integer values introduce.

We encode:
- `month` (1–12) → sin, cos
- `day_of_week` (0–6) → sin, cos
- `hour` (0–23) → sin, cos
- `winter`, `spring`, `summer`, `fall` (0/1) → sin, cos (treating each as a category of 4 seasons)
- `year_day` (1–366) → sin, cos


In [7]:

# Training set cyclic features
cyclic_train = train_set[['month', 'day_of_week', 'hour', 'winter', 'spring', 'summer', 'fall', 'year_day']].values

# Helper: create sin/cos columns
def cyclic_encode(data, period):
    return np.sin(2 * np.pi * data / period), np.cos(2 * np.pi * data / period)

sin_month, cos_month = cyclic_encode(cyclic_train[:, 0:1], 12)
sin_week, cos_week   = cyclic_encode(cyclic_train[:, 1:2], 6)
sin_hour, cos_hour   = cyclic_encode(cyclic_train[:, 2:3], 24)
sin_winter, cos_winter = cyclic_encode(cyclic_train[:, 3:4], 4)
sin_spring, cos_spring = cyclic_encode(cyclic_train[:, 4:5], 4)
sin_summer, cos_summer = cyclic_encode(cyclic_train[:, 5:6], 4)
sin_fall, cos_fall     = cyclic_encode(cyclic_train[:, 6:7], 4)
sin_year_day, cos_year_day = cyclic_encode(cyclic_train[:, 7:8], 365)

train_cyclic = np.concatenate((
    sin_month, cos_month,
    sin_week, cos_week,
    sin_hour, cos_hour,
    sin_winter, cos_winter,
    sin_spring, cos_spring,
    sin_summer, cos_summer,
    sin_fall, cos_fall,
    sin_year_day, cos_year_day
), axis=1)
print("Train cyclic shape:", train_cyclic.shape)

# Validation set
cyclic_val = validation_set[['month', 'day_of_week', 'hour', 'winter', 'spring', 'summer', 'fall', 'year_day']].values
sin_month_v, cos_month_v = cyclic_encode(cyclic_val[:, 0:1], 12)
sin_week_v, cos_week_v   = cyclic_encode(cyclic_val[:, 1:2], 6)
sin_hour_v, cos_hour_v   = cyclic_encode(cyclic_val[:, 2:3], 24)
sin_winter_v, cos_winter_v = cyclic_encode(cyclic_val[:, 3:4], 4)
sin_spring_v, cos_spring_v = cyclic_encode(cyclic_val[:, 4:5], 4)
sin_summer_v, cos_summer_v = cyclic_encode(cyclic_val[:, 5:6], 4)
sin_fall_v, cos_fall_v     = cyclic_encode(cyclic_val[:, 6:7], 4)
sin_year_day_v, cos_year_day_v = cyclic_encode(cyclic_val[:, 7:8], 365)

validation_cyclic = np.concatenate((
    sin_month_v, cos_month_v,
    sin_week_v, cos_week_v,
    sin_hour_v, cos_hour_v,
    sin_winter_v, cos_winter_v,
    sin_spring_v, cos_spring_v,
    sin_summer_v, cos_summer_v,
    sin_fall_v, cos_fall_v,
    sin_year_day_v, cos_year_day_v
), axis=1)

# Test set
cyclic_test = test_set[['month', 'day_of_week', 'hour', 'winter', 'spring', 'summer', 'fall', 'year_day']].values
sin_month_t, cos_month_t = cyclic_encode(cyclic_test[:, 0:1], 12)
sin_week_t, cos_week_t   = cyclic_encode(cyclic_test[:, 1:2], 6)
sin_hour_t, cos_hour_t   = cyclic_encode(cyclic_test[:, 2:3], 24)
sin_winter_t, cos_winter_t = cyclic_encode(cyclic_test[:, 3:4], 4)
sin_spring_t, cos_spring_t = cyclic_encode(cyclic_test[:, 4:5], 4)
sin_summer_t, cos_summer_t = cyclic_encode(cyclic_test[:, 5:6], 4)
sin_fall_t, cos_fall_t     = cyclic_encode(cyclic_test[:, 6:7], 4)
sin_year_day_t, cos_year_day_t = cyclic_encode(cyclic_test[:, 7:8], 365)

test_cyclic = np.concatenate((
    sin_month_t, cos_month_t,
    sin_week_t, cos_week_t,
    sin_hour_t, cos_hour_t,
    sin_winter_t, cos_winter_t,
    sin_spring_t, cos_spring_t,
    sin_summer_t, cos_summer_t,
    sin_fall_t, cos_fall_t,
    sin_year_day_t, cos_year_day_t
), axis=1)

print("Validation cyclic shape:", validation_cyclic.shape)
print("Test cyclic shape:", test_cyclic.shape)


Train cyclic shape: (84907, 16)
Validation cyclic shape: (24259, 16)
Test cyclic shape: (12130, 16)



## 6. Combine All Transformed Features
We concatenate the normalized target, one‑hot encoded binary features, and cyclic encoded features into final arrays for model input.


In [8]:

# Training
train_final = np.concatenate((scaled_train_set_load, train_categorical, train_cyclic), axis=1)
print("Train final shape:", train_final.shape)

# Validation
validation_final = np.concatenate((scaled_validation_set_load, validation_categorical, validation_cyclic), axis=1)
print("Validation final shape:", validation_final.shape)

# Test
test_final = np.concatenate((scaled_test_set_load, test_categorical, test_cyclic), axis=1)
print("Test final shape:", test_final.shape)


Train final shape: (84907, 21)
Validation final shape: (24259, 21)
Test final shape: (12130, 21)



## 7. Save to CSV Files
We create DataFrames with appropriate column names and save them for later model training.


In [9]:

# Define column names
feature_names = [
    'aep',
    'Is_holiday0', 'Is_holiday1',
    'Is_weekend0', 'Is_weekend1',
    'sin_month', 'cos_month',
    'sin_week', 'cos_week',
    'sin_hour', 'cos_hour',
    'sin_winter', 'cos_winter',
    'sin_spring', 'cos_spring',
    'sin_summer', 'cos_summer',
    'sin_fall', 'cos_fall',
    'sin_year_day', 'cos_year_day'
]

# Create DataFrames
train_df = pd.DataFrame(train_final, columns=feature_names)
validation_df = pd.DataFrame(validation_final, columns=feature_names)
test_df = pd.DataFrame(test_final, columns=feature_names)

# Save to CSV
train_df.to_csv('7_AEP_train.csv', index=False)
validation_df.to_csv('8_AEP_validation.csv', index=False)
test_df.to_csv('9_AEP_test.csv', index=False)

print("Train saved: 7_AEP_train.csv")
print("Validation saved: 8_AEP_validation.csv")
print("Test saved: 9_AEP_test.csv")


Train saved: 7_AEP_train.csv
Validation saved: 8_AEP_validation.csv
Test saved: 9_AEP_test.csv



## 8. Verification
Check the shapes and a few rows of the saved files to ensure correctness.


In [10]:

print("Train shape:", train_df.shape)
train_df.head(2)


Train shape: (84907, 21)


,aep,Is_holiday0,Is_holiday1,Is_weekend0,Is_weekend1,sin_month,cos_month,sin_week,cos_week,sin_hour,...,sin_winter,cos_winter,sin_spring,cos_spring,sin_summer,cos_summer,sin_fall,cos_fall,sin_year_day,cos_year_day
0,0.210289,1.0,0.0,1.0,0.0,-0.866025,0.5,-0.866025,-0.5,0.258819,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,6.123234e-17,-0.999769,0.021516
1,0.175836,1.0,0.0,1.0,0.0,-0.866025,0.5,-0.866025,-0.5,0.500000,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,6.123234e-17,-0.999769,0.021516


In [11]:

print("Validation shape:", validation_df.shape)
validation_df.head(2)


Validation shape: (24259, 21)


,aep,Is_holiday0,Is_holiday1,Is_weekend0,Is_weekend1,sin_month,cos_month,sin_week,cos_week,sin_hour,...,sin_winter,cos_winter,sin_spring,cos_spring,sin_summer,cos_summer,sin_fall,cos_fall,sin_year_day,cos_year_day
0,0.351051,1.0,0.0,0.0,1.0,1.224647e-16,-1.0,-2.449294e-16,1.0,-0.866025,...,0.0,1.0,0.0,1.0,1.0,6.123234e-17,0.0,1.0,0.39359,-0.919286
1,0.337782,1.0,0.0,0.0,1.0,1.224647e-16,-1.0,-2.449294e-16,1.0,-0.707107,...,0.0,1.0,0.0,1.0,1.0,6.123234e-17,0.0,1.0,0.39359,-0.919286


In [12]:

print("Test shape:", test_df.shape)
test_df.head(2)


Test shape: (12130, 21)


,aep,Is_holiday0,Is_holiday1,Is_weekend0,Is_weekend1,sin_month,cos_month,sin_week,cos_week,sin_hour,...,sin_winter,cos_winter,sin_spring,cos_spring,sin_summer,cos_summer,sin_fall,cos_fall,sin_year_day,cos_year_day
0,0.644836,1.0,0.0,1.0,0.0,1.0,6.123234e-17,0.866025,-0.5,-0.707107,...,0.0,1.0,1.0,6.123234e-17,0.0,1.0,0.0,1.0,0.956235,0.2926
1,0.613021,1.0,0.0,1.0,0.0,1.0,6.123234e-17,0.866025,-0.5,-0.866025,...,0.0,1.0,1.0,6.123234e-17,0.0,1.0,0.0,1.0,0.956235,0.2926



## Summary

- The dataset was split chronologically into training, validation, and test sets.
- The target `aep` was normalized using Min‑Max scaling.
- Binary features `holiday` and `weekend` were one‑hot encoded.
- Periodic features (`hour`, `month`, `day_of_week`, `year_day`, seasons) were cyclically encoded using sine and cosine transformations.
- The final feature sets have **21 columns** (1 target + 4 one‑hot + 16 cyclic).
- All processed datasets are saved as CSV files ready for model training.

**Prepared by:** Shaheer Khan (22JZELE0457)  
